In [ ]:
import pandas as pd
import numpy as np
from datetime import date

# Load dataset 2 (2020-2025)
df2 = pd.read_csv("online_sales_dataset_for_2020-2025.csv")

# -----------------------
# FIX COLUMN NAMES
# -----------------------
df2.rename(columns={
    "InvoiceDate": "InvoiceDate",
    "UnitPrice": "UnitPrice",
    "Quantity": "Quantity",
    "Discount": "Discount",
    "ShippingCost": "ShippingCost",
    "Category": "Category",
    "Description": "ProductName",
    "PaymentMethod": "PaymentMethod",
    "OrderPriority": "OrderPriority",
    "ReturnStatus": "ReturnStatus",
    "Country": "Country",
    "SalesChannel": "SalesChannel"
}, inplace=True)

# -----------------------
# FIX DATE COLUMN (DATE ONLY)
# -----------------------
# Convert to datetime
df2["InvoiceDate"] = pd.to_datetime(df2["InvoiceDate"], errors="coerce")

# Extract ONLY the date (datetime.date)
df2["Date"] = df2["InvoiceDate"].dt.date

# Drop rows where date could not be parsed
df2 = df2.dropna(subset=["Date"])

# -----------------------
# KEEP ONLY 2020–2025
# -----------------------
df2 = df2[
    (df2["Date"] >= date(2020, 1, 1)) &
    (df2["Date"] <= date(2025, 12, 31))
]

# -----------------------
# REMOVE INVALID ROWS
# -----------------------
df2 = df2[(df2["Quantity"] > 0) & (df2["UnitPrice"] > 0)]

# -----------------------
# CREATE RETURN BINARY
# -----------------------
df2["ReturnBinary"] = np.where(df2["ReturnStatus"] == "Returned", 1, 0)

# -----------------------
# NUMERICAL FEATURES
# -----------------------

# Revenue
df2["Revenue"] = df2["UnitPrice"] * df2["Quantity"]

# Discount amount (discount is a percentage)
df2["Discount_Amount"] = df2["Revenue"] * df2["Discount"]

# Net revenue
df2["Net_Revenue"] = df2["Revenue"] - df2["Discount_Amount"]

# Estimated Profit
df2["Estimated_Profit"] = df2["Net_Revenue"] - df2["ShippingCost"]

# Profit per unit
df2["Profit_per_unit"] = df2["Estimated_Profit"] / df2["Quantity"]

# Effective Unit Price
df2["Effective_UnitPrice"] = df2["Net_Revenue"] / df2["Quantity"]

# Shipping burden
df2["Shipping_Burden"] = df2["ShippingCost"] / df2["Net_Revenue"]

# Order value
df2["Order_Value"] = df2["Net_Revenue"] + df2["ShippingCost"]

# Discount ratio
df2["Discount_Ratio"] = df2["Discount_Amount"] / df2["Revenue"]

df2.shape
df2.head()


,InvoiceNo,StockCode,ProductName,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Discount,PaymentMethod,...,ReturnBinary,Revenue,Discount_Amount,Net_Revenue,Estimated_Profit,Profit_per_unit,Effective_UnitPrice,Shipping_Burden,Order_Value,Discount_Ratio
0,221958,SKU_1964,White Mug,38,2020-01-01 00:00:00,1.71,37039.0,Australia,0.47,Bank Transfer,...,0,64.98,30.5406,34.4394,23.6494,0.622353,0.9063,0.313304,45.2294,0.47
1,771155,SKU_1241,White Mug,18,2020-01-01 01:00:00,41.25,19144.0,Spain,0.19,paypall,...,0,742.50,141.0750,601.4250,591.9150,32.884167,33.4125,0.015812,610.9350,0.19
2,231932,SKU_1501,Headphones,49,2020-01-01 02:00:00,29.11,50472.0,Germany,0.35,Bank Transfer,...,1,1426.39,499.2365,927.1535,904.1235,18.451500,18.9215,0.024839,950.1835,0.35
3,465838,SKU_1760,Desk Lamp,14,2020-01-01 03:00:00,76.68,96586.0,Netherlands,0.14,paypall,...,0,1073.52,150.2928,923.2272,912.1472,65.153371,65.9448,0.012001,934.3072,0.14
5,744167,SKU_1006,Office Chair,47,2020-01-01 05:00:00,70.16,53887.0,Sweden,0.48,Credit Card,...,0,3297.52,1582.8096,1714.7104,1700.7304,36.185753,36.4832,0.008153,1728.6904,0.48


In [ ]:
import pandas as pd
import numpy as np

# Load dataset 1 (2018)
df1 = pd.read_csv("E-commerce Dataset.csv")

# -----------------------
# RENAME COLUMNS TO MATCH DF2
# -----------------------
df1.rename(columns={
    "Order_Date": "OrderDateRaw",
    "Sales": "UnitPrice",
    "Quantity": "Quantity",
    "Discount": "Discount",
    "Shipping_Cost": "ShippingCost",
    "Product_Category": "Category",
    "Product": "ProductName",
    "Payment_method": "PaymentMethod",
    "Order_Priority": "OrderPriority",
    "Profit": "Profit_IGNORE"
}, inplace=True)

# -----------------------
# FIX DATE COLUMN (DATE ONLY)
# -----------------------
df1["OrderDateRaw"] = pd.to_datetime(df1["OrderDateRaw"], errors="coerce")
df1["Date"] = df1["OrderDateRaw"].dt.date     # SAME TYPE as df2 Date

df1.drop(columns=["OrderDateRaw"], inplace=True)

df1 = df1.dropna(subset=["Date"]).sort_values(by="Date").reset_index(drop=True)

# -----------------------
# REMOVE INVALID ROWS
# -----------------------
df1 = df1[(df1["Quantity"] > 0) & (df1["UnitPrice"] > 0)]

# -----------------------
# ADD MISSING COLUMNS (TO MATCH DF2)
# -----------------------
df1["ReturnStatus"] = "Not Available"
df1["ReturnBinary"] = 0
df1["Country"] = "Not Available"

# -----------------------
# NUMERICAL FEATURES (SAME AS DF2)
# -----------------------
df1["Revenue"] = df1["UnitPrice"] * df1["Quantity"]
df1["Discount_Amount"] = df1["Revenue"] * df1["Discount"]
df1["Net_Revenue"] = df1["Revenue"] - df1["Discount_Amount"]
df1["Estimated_Profit"] = df1["Net_Revenue"] - df1["ShippingCost"]
df1["Profit_per_unit"] = df1["Estimated_Profit"] / df1["Quantity"]
df1["Effective_UnitPrice"] = df1["Net_Revenue"] / df1["Quantity"]
df1["Shipping_Burden"] = df1["ShippingCost"] / df1["Net_Revenue"]
df1["Order_Value"] = df1["Net_Revenue"] + df1["ShippingCost"]
df1["Discount_Ratio"] = df1["Discount_Amount"] / df1["Revenue"]

# -----------------------
# DROP UNUSED COLUMN
# -----------------------
df1.drop(columns=["Profit_IGNORE"], inplace=True)

# DONE
df1.head()


,Time,Aging,Customer_Id,Gender,Device_Type,Customer_Login_type,Category,ProductName,UnitPrice,Quantity,...,Country,Revenue,Discount_Amount,Net_Revenue,Estimated_Profit,Profit_per_unit,Effective_UnitPrice,Shipping_Burden,Order_Value,Discount_Ratio
0,18:55:23,6.0,93045,Female,Web,Member,Home & Furniture,Towels,228.0,2.0,...,Not Available,456.0,91.2,364.8,351.8,175.900,182.4,0.035636,377.8,0.2
1,16:34:38,5.0,84298,Female,Web,Member,Home & Furniture,Dinner Crockery,133.0,1.0,...,Not Available,133.0,53.2,79.8,75.6,75.600,79.8,0.052632,84.0,0.4
2,21:45:41,10.0,83120,Female,Web,Member,Home & Furniture,Sofas,67.0,4.0,...,Not Available,268.0,107.2,160.8,159.1,39.775,40.2,0.010572,162.5,0.4
3,21:06:02,2.0,63293,Male,Web,Member,Home & Furniture,Sofa Covers,216.0,4.0,...,Not Available,864.0,172.8,691.2,678.5,169.625,172.8,0.018374,703.9,0.2
4,11:46:12,1.0,52631,Male,Web,Member,Auto & Accessories,Car Speakers,211.0,5.0,...,Not Available,1055.0,316.5,738.5,729.6,145.920,147.7,0.012051,747.4,0.3


In [ ]:
#Merge
df_all = pd.concat([df1, df2], ignore_index=True)

cols_to_keep = [
    "Date", "Country", "Category", "ProductName", "PaymentMethod", "OrderPriority",
    "ReturnStatus", "ReturnBinary",
    "UnitPrice", "Quantity", "Discount", "ShippingCost",
    "Revenue", "Discount_Amount", "Net_Revenue", "Estimated_Profit",
    "Profit_per_unit", "Effective_UnitPrice", "Shipping_Burden",
    "Order_Value"]

df_all = df_all[cols_to_keep]

df_all = df_all.sort_values(by="Date").reset_index(drop=True)

df_all.shape

(98580, 20)

In [77]:
covid = pd.read_csv("covid_policy-lockdown_tracker.csv")

# STEP 2 — Fix date column
covid["Date"] = pd.to_datetime(covid["date"], errors="coerce").dt.date

# Drop the old 'date' column
covid.drop(columns=["date"], inplace=True)

# STEP 3 — Rename country column
covid.rename(columns={"countryname": "Country"}, inplace=True)

# STEP 4 — Rename stay-home column
covid.rename(columns={
    "c6_stay_at_home_requirements": "stay_home_requirement",
}, inplace=True)

# STEP 5 — Keep only useful COVID columns
covid = covid[["Country", "Date", "stringencyindex", "stay_home_requirement"]]

# STEP 6 — Use only countries from df2 (2020–2025)
valid_countries = df2["Country"].unique()

# Filter covid dataset to only the needed countries
covid = covid[covid["Country"].isin(valid_countries)]

covid = covid.groupby(["Country", "Date"], as_index=False).agg({
    "stringencyindex": "mean",
    "stay_home_requirement": "max"
})

# STEP 7 — MERGE COVID WITH SALES DATA
final = df_all.merge(covid, on=["Country", "Date"], how="left")

# STEP 8 — Fill 2018 and unmatched rows with 0
final["stringencyindex"] = final["stringencyindex"].fillna(0)
final["stay_home_requirement"] = final["stay_home_requirement"].fillna(0)

# STEP 9 — Create COVID flags

# 1. is_covid (year >= 2020)
final["is_covid"] = (final["stringencyindex"] > 0).astype(int)

# 2. is_lockdown (stay_home_requirement > 0)
final["is_lockdown"] = (final["stay_home_requirement"] > 0).astype(int)


final.to_csv("final_merged.csv", index=False)
